<a href="https://colab.research.google.com/github/cto-school/mathematics-for-machine-learning/blob/main/04-linear-algebra-2-matrices/04b_linear_systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 04b — Linear Systems: Solving $A\mathbf{x}=\mathbf{b}$

A **linear system** is a set of linear equations, for example

$$ \begin{aligned} 2x + y &= 5,\\ x - y &= 1. \end{aligned} $$

We can pack it into a single matrix equation $A\mathbf{x}=\mathbf{b}$:

$$ \underbrace{\begin{bmatrix} 2 & 1 \\ 1 & -1 \end{bmatrix}}_{A}
   \underbrace{\begin{bmatrix} x \\ y \end{bmatrix}}_{\mathbf{x}}
   = \underbrace{\begin{bmatrix} 5 \\ 1 \end{bmatrix}}_{\mathbf{b}}. $$

In this notebook we solve such systems with NumPy, meet the **inverse**,
**determinant**, and **rank**, and see the *geometry*: solutions are where lines
intersect.

## 1. Solving a system with `np.linalg.solve`

Give NumPy the matrix $A$ and the right-hand side $\mathbf{b}$; it returns the
$\mathbf{x}$ that solves $A\mathbf{x}=\mathbf{b}$.

In [ ]:
import numpy as np

A = np.array([[2.0, 1.0],
              [1.0, -1.0]])
b = np.array([5.0, 1.0])

x = np.linalg.solve(A, b)
print("solution x =", x)          # [2. 1.]  ->  x=2, y=1

# Always sanity-check: does A @ x give back b?
print("A @ x =", A @ x)           # [5. 1.]
print("matches b? ", np.allclose(A @ x, b))

## 2. The inverse $A^{-1}$ — and why `solve` is better

If $A$ is square and "nice", there is an **inverse** $A^{-1}$ with
$A^{-1}A = AA^{-1} = I$. Then in principle $\mathbf{x}=A^{-1}\mathbf{b}$.
NumPy gives it with `np.linalg.inv`.

In [ ]:
import numpy as np
A = np.array([[2.0, 1.0],
              [1.0, -1.0]])
b = np.array([5.0, 1.0])

Ainv = np.linalg.inv(A)
print("A^-1 =\n", Ainv)
print("A^-1 @ A =\n", np.round(Ainv @ A, 10))   # the identity

x = Ainv @ b
print("x via inverse =", x)        # same [2. 1.]

> **Why prefer `solve` over `inv`?** Computing the full inverse does *more*
> work than needed and is **less accurate** (it accumulates rounding error).
> `np.linalg.solve(A, b)` goes straight to the answer faster and more reliably.
> **Rule of thumb: never write `inv(A) @ b`; write `solve(A, b)`.**

In [ ]:
import numpy as np
A = np.array([[2.0, 1.0],
              [1.0, -1.0]])
b = np.array([5.0, 1.0])

print("solve :", np.linalg.solve(A, b))   # preferred
print("inv   :", np.linalg.inv(A) @ b)    # works, but discouraged

## 3. The determinant: area/volume scaling

The **determinant** $\det(A)$ measures how much the transformation $A$ scales
**area** (in 2-D) or **volume** (in 3-D). A unit square has area 1; after
applying $A$ its area becomes $|\det(A)|$.

- $|\det A| > 1$: the map *expands* area.
- $|\det A| < 1$: it *shrinks* area.
- $\det A < 0$: it also *flips* orientation (a mirror reflection).

In [ ]:
import numpy as np

A = np.array([[2.0, 0.0],
              [0.0, 3.0]])     # stretches x by 2, y by 3
print("det(A) =", np.linalg.det(A))   # 6.0  -> area multiplied by 6

R = np.array([[0.0, -1.0],
              [1.0,  0.0]])    # a 90-degree rotation
print("det(R) =", np.linalg.det(R))   # 1.0  -> rotations preserve area

### Visual check: area really scales by $|\det A|$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

square = np.array([[0, 1, 1, 0, 0],
                   [0, 0, 1, 1, 0]])
A = np.array([[2.0, 1.0],
              [0.0, 1.5]])
out = A @ square

print("det(A) =", np.linalg.det(A))   # 3.0 -> new area is 3

plt.figure(figsize=(5, 5))
plt.fill(square[0], square[1], alpha=0.3, label="unit square (area 1)")
plt.fill(out[0],    out[1],    alpha=0.3, label="image (area = |det A|)")
plt.gca().set_aspect("equal"); plt.grid(True); plt.legend()
plt.title("Determinant = area-scaling factor")
plt.show()

## 4. Singular matrices: when $\det A = 0$

If $\det(A)=0$, the matrix **collapses** area to zero — it squashes the plane
onto a line (or a point). Such a matrix is called **singular**: it has **no
inverse**, and $A\mathbf{x}=\mathbf{b}$ may have *no* solution or *infinitely
many*. `np.linalg.solve` will refuse (it raises an error).

In [ ]:
import numpy as np

A = np.array([[1.0, 2.0],
              [2.0, 4.0]])     # row 2 is exactly 2 x row 1
print("det(A) =", np.linalg.det(A))   # 0.0 (up to tiny rounding) -> singular

try:
    np.linalg.inv(A)
except np.linalg.LinAlgError as e:
    print("inv failed:", e)

try:
    np.linalg.solve(A, np.array([1.0, 0.0]))
except np.linalg.LinAlgError as e:
    print("solve failed:", e)

## 5. Rank: how many independent directions

The **rank** of $A$ is the number of linearly independent rows (equivalently,
columns) — how many genuinely different directions the matrix keeps. For a
square $n\times n$ matrix:

- **full rank** ($\text{rank}=n$): invertible, $\det\neq 0$, unique solution.
- **rank $< n$**: singular, $\det = 0$, no/infinitely-many solutions.

Use `np.linalg.matrix_rank`.

In [ ]:
import numpy as np

full = np.array([[2.0, 1.0],
                 [1.0, -1.0]])
deficient = np.array([[1.0, 2.0],
                      [2.0, 4.0]])   # second row depends on the first

print("rank(full)      =", np.linalg.matrix_rank(full))       # 2  (full rank)
print("rank(deficient) =", np.linalg.matrix_rank(deficient))  # 1  (rank deficient)

## 6. Geometry: no solution vs. infinitely many

Each equation $a x + b y = c$ is a **line** in the plane. Solving a $2\times 2$
system means finding where the two lines **meet**.

- **One solution:** the lines cross at a single point (the normal case).
- **No solution:** the lines are **parallel but different** — they never meet.
- **Infinitely many:** the two equations describe the **same line**.

Let's draw all three.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-1, 4, 100)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# --- (a) Unique solution: 2x+y=5 and x-y=1 cross at (2,1) ---
axes[0].plot(x, 5 - 2*x, label="2x + y = 5")
axes[0].plot(x, x - 1,   label="x - y = 1")
axes[0].plot(2, 1, "ko")               # the intersection
axes[0].set_title("Unique solution")

# --- (b) No solution: two PARALLEL lines (same slope, different intercept) ---
axes[1].plot(x, 2 - x,     label="x + y = 2")
axes[1].plot(x, 4 - x,     label="x + y = 4")
axes[1].set_title("No solution (parallel)")

# --- (c) Infinitely many: the SAME line written twice ---
axes[2].plot(x, 2 - x,        label="x + y = 2")
axes[2].plot(x, 2 - x, "--",  label="2x + 2y = 4 (same line)")
axes[2].set_title("Infinitely many (identical)")

for ax in axes:
    ax.set_aspect("equal"); ax.grid(True); ax.legend(); ax.set_xlim(-1, 4)
plt.show()

We can detect these cases numerically with the **determinant** and **rank** of
$A$ (and of the *augmented* matrix $[A\,|\,\mathbf{b}]$):

In [ ]:
import numpy as np

# No-solution system (parallel lines): x+y=2, x+y=4
A = np.array([[1.0, 1.0],
              [1.0, 1.0]])
b = np.array([2.0, 4.0])

print("det(A)            =", np.linalg.det(A))            # 0 -> singular
print("rank(A)           =", np.linalg.matrix_rank(A))    # 1
aug = np.column_stack([A, b])                             # [A | b]
print("rank([A|b])       =", np.linalg.matrix_rank(aug))  # 2
print("rank(A) < rank([A|b])  -> NO solution")

## 7. A fully worked $2\times 2$ example, visualized

Solve

$$ \begin{aligned} 3x + 2y &= 12,\\ x - y &= 1, \end{aligned} $$

both algebraically (`solve`) and graphically (intersection of two lines).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.array([[3.0, 2.0],
              [1.0, -1.0]])
b = np.array([12.0, 1.0])

# 1) Solve it
sol = np.linalg.solve(A, b)
print("solution (x, y) =", sol)          # [2.8 1.8]
print("check A @ x:", A @ sol, " vs b:", b)

# 2) Show it as intersecting lines
x = np.linspace(-1, 5, 100)
line1 = (12 - 3*x) / 2      # from 3x + 2y = 12  ->  y = (12 - 3x)/2
line2 = x - 1               # from  x -  y = 1   ->  y = x - 1

plt.figure(figsize=(5, 5))
plt.plot(x, line1, label="3x + 2y = 12")
plt.plot(x, line2, label="x - y = 1")
plt.plot(sol[0], sol[1], "ko", markersize=9, label=f"solution ({sol[0]:.1f}, {sol[1]:.1f})")
plt.axhline(0, color="gray", lw=0.5); plt.axvline(0, color="gray", lw=0.5)
plt.gca().set_aspect("equal"); plt.grid(True); plt.legend()
plt.title("Solution = where the two lines cross")
plt.show()

---
## ✍️ Exercise 1 (solution included)

Solve the system

$$ \begin{aligned} 4x + 3y &= 10,\\ 2x - y &= 0, \end{aligned} $$

using `np.linalg.solve`. Verify your answer by plugging it back in, and report
$\det(A)$ to confirm the system has a unique solution.

**Solution:**

In [ ]:
import numpy as np
A = np.array([[4.0, 3.0],
              [2.0, -1.0]])
b = np.array([10.0, 0.0])

x = np.linalg.solve(A, b)
print("solution =", x)                 # [1. 2.]
print("check A @ x =", A @ x)          # [10. 0.] -> matches b
print("det(A) =", np.linalg.det(A))    # -10 (nonzero) -> unique solution

## ✍️ Exercise 2 (solution included)

Consider $A = \begin{bmatrix} 1 & 2 \\ 2 & 4 \end{bmatrix}$. Compute
$\det(A)$ and $\text{rank}(A)$. Is $A$ invertible? Then decide, using ranks,
whether $A\mathbf{x} = \begin{bmatrix} 3 \\ 6 \end{bmatrix}$ has solutions, and
whether $A\mathbf{x} = \begin{bmatrix} 3 \\ 7 \end{bmatrix}$ does.

**Solution:**

In [ ]:
import numpy as np
A = np.array([[1.0, 2.0],
              [2.0, 4.0]])

print("det(A)  =", np.linalg.det(A))            # 0 -> singular, NOT invertible
print("rank(A) =", np.linalg.matrix_rank(A))    # 1

# Case b = [3, 6]: it lies ON the line -> infinitely many solutions
b1 = np.array([3.0, 6.0])
aug1 = np.column_stack([A, b1])
print("b=[3,6]: rank([A|b]) =", np.linalg.matrix_rank(aug1),
      "== rank(A) -> infinitely many solutions")

# Case b = [3, 7]: inconsistent -> no solution
b2 = np.array([3.0, 7.0])
aug2 = np.column_stack([A, b2])
print("b=[3,7]: rank([A|b]) =", np.linalg.matrix_rank(aug2),
      ">  rank(A) -> NO solution")

---
## 📝 Homework (no solutions provided)

1. Solve $\begin{aligned} x + 2y &= 4 \\ 3x - y &= 5 \end{aligned}$ with
   `np.linalg.solve`, verify by substitution, and plot the two lines with their
   intersection marked.
2. For $A = \begin{bmatrix} 1 & 1 \\ 1 & 1.0001 \end{bmatrix}$, compute
   $\det(A)$. It is *tiny* but nonzero — such a matrix is "nearly singular".
   Solve $A\mathbf{x}=(2,2)$ and then $A\mathbf{x}=(2,2.001)$; notice how a tiny
   change in $\mathbf{b}$ swings the answer a lot.
3. Build a $3\times 3$ system of your choice that has a unique solution
   (check $\det \neq 0$), solve it, and verify $A\mathbf{x}=\mathbf{b}$.
4. Construct a $2\times 2$ system whose two lines are **parallel** (no
   solution). Confirm with `det`, `matrix_rank(A)`, and
   `matrix_rank([A | b])`, and plot the two parallel lines.